# Tomahawk Rev2.2 — Investor-facing Kaggle computational validation

This notebook is deliberately an **external-input audit**. It does not generate, tune, or alter model inputs. Upload the accompanying CSV/JSON files as one Kaggle Dataset, then use **Run All**.

It verifies implementation bookkeeping, Bosch–Hale D–T reference-rate values, radial quadrature convergence, deterministic repeatability, input-file hashes, and conditional scenario pass rates. It is not experimental validation and cannot prove reactor or hardware performance.

The Kaggle GPU is not required: the evaluator is vectorized NumPy and is intentionally kept CPU-friendly so the run is fast and reproducible.

Reference-rate source: [UWFDM-1268, Table 6](https://fti.neep.wisc.edu/fti.neep.wisc.edu/pdf/fdm1268.pdf), citing Bosch and Hale, *Nuclear Fusion* 32 (1992) 611.


In [ ]:
# Tomahawk Rev2.2 — Investor-facing computational validation package
#
# This notebook cell deliberately consumes external CSV files. It does not
# generate, tune, or alter model inputs. The calculations are a frozen 0-D
# screen; the output separates implementation verification from scenario
# robustness and states what still requires physical validation.

from datetime import datetime, timezone
import hashlib
import json
import math
import os
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


MODEL_VERSION = "Tomahawk Rev2.2 frozen external-input evaluator"
MODEL_REVISION = "REV22-FROZEN-20260825"
MODEL_SPEC_TEXT = (
    "Bosch-Hale DT reactivity in m^3/s; 32-point radial Gauss-Legendre "
    "volume weighting; nD*nT*<sigma-v>*950 m^3; 17.6 MeV/fusion; "
    "alpha=3.5/17.6; neutron=14.1/17.6; laser=2.50 MW/module-shot "
    "times repetition rate; gates are fixed at the Rev2.2 thresholds."
)
MODEL_SPEC_SHA256 = hashlib.sha256(MODEL_SPEC_TEXT.encode("utf-8")).hexdigest()

PARAMETER_NAMES = [
    "gamma_n", "density_scale", "fuel_fraction", "deuterium_fraction",
    "Ti0_keV", "Ti_edge_keV", "gamma_T", "Te_to_Ti", "B0_T", "tau_E_s",
    "Z_eff", "alpha_deposition", "laser_coupling", "repetition_rate_Hz",
    "wallplug_efficiency", "neutron_heat_capture", "alpha_heat_capture",
    "high_grade_heat_fraction", "eta_high_grade", "eta_low_grade",
    "nonlaser_aux_GWe", "TBR", "tritium_process_eff",
]

DATASETS = [
    {
        "role": "broad_global_stress",
        "filename": "Tomahawk_Rev22_Broad_Input_Dataset.csv",
        "description": "broad global stress distribution; not expected to pass most gates",
    },
    {
        "role": "independent_computational_holdout",
        "filename": "Tomahawk_Rev22_Independent_Holdout_Input_Dataset.csv",
        "description": "space-filling broad-range holdout for software/model reproducibility",
    },
    {
        "role": "controlled_envelope_v2",
        "filename": "Tomahawk_Rev22_Controlled_Envelope_v2_Input_Dataset.csv",
        "description": "proposed design/tolerance envelope; not measured uncertainty",
    },
    {
        "role": "multifactor_improved_envelope",
        "filename": "Tomahawk_Rev22_Multifactor_Improved_Envelope_Input_Dataset.csv",
        "description": "proposed improvement scenario; not measured hardware performance",
    },
]

OUT_DIR = (
    Path("/kaggle/working/tomahawk_rev22_investor_validation")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "tomahawk_rev22_investor_validation"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
N_QUAD = int(os.environ.get("TOMAHAWK_VALIDATION_QUADRATURE", "32"))
SAVE_FULL_CASE_OUTPUT = os.environ.get("TOMAHAWK_SAVE_FULL_CASE_OUTPUT", "0") == "1"
AUDIT_SAMPLE_PER_DATASET = 100


def locate_file(filename):
    requested_root = os.environ.get("TOMAHAWK_INPUT_ROOT", "").strip()
    roots = []
    if requested_root:
        roots.append(Path(requested_root).expanduser())
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.append(Path.cwd())
    for root in roots:
        direct = root / filename
        if direct.exists():
            return direct.resolve()
        if root.exists() and root.is_dir():
            matches = sorted(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Missing {filename}. Upload the CSV files as a Kaggle Dataset, or set "
        "TOMAHAWK_INPUT_ROOT to their directory."
    )


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_optional_spec(csv_path):
    candidates = [
        csv_path.with_name(csv_path.name.replace("_Dataset.csv", "_Spec.json")),
        csv_path.with_name(csv_path.stem + "_Spec.json"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    try:
        return locate_file(csv_path.name.replace("_Dataset.csv", "_Spec.json"))
    except FileNotFoundError:
        return None


def bosch_hale_dt_reactivity(T):
    """Bosch-Hale D-T reactivity in m^3/s for T in keV."""
    T = np.clip(np.asarray(T, dtype=np.float64), 0.2, 100.0)
    BG = 34.3827
    mrc2 = 1.124656e6
    C1, C2, C3, C4 = 1.17302e-9, 1.51361e-2, 7.51886e-2, 4.60643e-3
    C5, C6, C7 = 1.35000e-2, -1.06750e-4, 1.36600e-5
    numerator = T * (C2 + T * (C4 + T * C6))
    denominator = 1.0 + T * (C3 + T * (C5 + T * C7))
    theta = T / (1.0 - numerator / denominator)
    xi = (BG ** 2 / (4.0 * theta)) ** (1.0 / 3.0)
    # Bosch-Hale coefficients give cm^3/s; convert to m^3/s for SI densities.
    return 1.0e-6 * C1 * theta * np.sqrt(xi / (mrc2 * T ** 3)) * np.exp(-3.0 * xi)


def quadrature(n_quad):
    legendre_x, legendre_w = np.polynomial.legendre.leggauss(int(n_quad))
    rho = 0.5 * (legendre_x + 1.0)
    weights = 0.5 * legendre_w * 2.0 * rho
    return rho, weights


MU0 = 4.0e-7 * math.pi
AMU = 1.66053906660e-27
KEV_TO_J = 1.0e3 * 1.602176634e-19
MEV_TO_J = 1.0e6 * 1.602176634e-19
M_T = 3.01604928199 * AMU
E_FUSION = 17.6 * MEV_TO_J
ALPHA_FRACTION = 3.5 / 17.6
NEUTRON_FRACTION = 14.1 / 17.6
VOLUME_M3 = 950.0
N_D0_BASE = 1.2e20
N_T0_BASE = 1.2e20
ACTIVE_MODULES = 10


def evaluate_batch(params, n_quad=32):
    """Frozen Rev2.2 equations; params contains equal-length NumPy arrays."""
    n_cases = len(next(iter(params.values())))
    gamma_n = params["gamma_n"]
    gamma_T = params["gamma_T"]
    total_ion_center = (N_D0_BASE + N_T0_BASE) * params["density_scale"]
    fuel_center = total_ion_center * params["fuel_fraction"]
    n_D0 = fuel_center * params["deuterium_fraction"]
    n_T0 = fuel_center * (1.0 - params["deuterium_fraction"])
    integral_reaction = np.zeros(n_cases, dtype=np.float64)
    integral_pressure = np.zeros(n_cases, dtype=np.float64)
    integral_brem_shape = np.zeros(n_cases, dtype=np.float64)
    rho_nodes, rho_weights = quadrature(n_quad)
    for rho, weight in zip(rho_nodes, rho_weights):
        base = max(0.0, 1.0 - float(rho) ** 2)
        density_shape = base ** gamma_n
        temperature_shape = base ** gamma_T
        Ti = params["Ti_edge_keV"] + (params["Ti0_keV"] - params["Ti_edge_keV"]) * temperature_shape
        Te = np.maximum(0.2, Ti * params["Te_to_Ti"])
        n_i = total_ion_center * density_shape
        n_D = n_D0 * density_shape
        n_T = n_T0 * density_shape
        integral_reaction += weight * n_D * n_T * bosch_hale_dt_reactivity(Ti)
        integral_pressure += weight * (n_i * Ti + n_i * Te) * KEV_TO_J
        integral_brem_shape += weight * n_i ** 2 * np.sqrt(Te)

    reaction_rate = VOLUME_M3 * integral_reaction
    fusion_W = reaction_rate * E_FUSION
    alpha_W = fusion_W * ALPHA_FRACTION
    neutron_W = fusion_W * NEUTRON_FRACTION
    alpha_deposited_W = alpha_W * params["alpha_deposition"]
    brem_W = VOLUME_M3 * 5.35e-37 * params["Z_eff"] * integral_brem_shape
    thermal_energy_J = 1.5 * VOLUME_M3 * integral_pressure
    transport_W = thermal_energy_J / params["tau_E_s"]
    laser_optical_W = 2.50e6 * params["repetition_rate_Hz"]
    laser_absorbed_W = laser_optical_W * params["laser_coupling"]
    self_ratio = alpha_deposited_W / np.maximum(brem_W + transport_W, 1.0)
    target_gain = (fusion_W / params["repetition_rate_Hz"]) / 2.50e6
    beta = 2.0 * MU0 * integral_pressure / params["B0_T"] ** 2
    captured_W = (
        neutron_W * params["neutron_heat_capture"]
        + alpha_W * params["alpha_heat_capture"]
        + laser_absorbed_W
    )
    high_grade_W = captured_W * params["high_grade_heat_fraction"]
    low_grade_W = captured_W - high_grade_W
    gross_module_W = params["eta_high_grade"] * high_grade_W + params["eta_low_grade"] * low_grade_W
    gross_hub_GWe = gross_module_W * ACTIVE_MODULES / 1.0e9
    laser_electric_hub_GWe = laser_optical_W / params["wallplug_efficiency"] * ACTIVE_MODULES / 1.0e9
    recirc_hub_GWe = laser_electric_hub_GWe + params["nonlaser_aux_GWe"]
    net_hub_GWe = gross_hub_GWe - recirc_hub_GWe
    t_burn_g_h = reaction_rate * M_T * 1000.0 * 3600.0 * ACTIVE_MODULES
    t_surplus_g_h = params["tritium_process_eff"] * params["TBR"] * t_burn_g_h - t_burn_g_h
    tau_required = np.where(
        alpha_deposited_W > brem_W,
        thermal_energy_J / np.maximum(alpha_deposited_W - brem_W, 1.0),
        np.inf,
    )
    tau_margin = params["tau_E_s"] / np.maximum(tau_required, 1.0e-30) - 1.0
    gate_arrays = {
        "fusion": fusion_W >= 5.0e9,
        "gain": target_gain >= 200.0,
        "wallplug": params["wallplug_efficiency"] >= 0.15,
        "self_heating": self_ratio >= 1.0,
        "TBR": params["TBR"] >= 1.15,
        "tritium_surplus": t_surplus_g_h >= 0.0,
        "captured_heat": captured_W >= 5.0e9,
        "gross_electric": gross_hub_GWe >= 20.0,
        "recirculating": recirc_hub_GWe <= 3.0,
        "net_electric": net_hub_GWe >= 17.0,
    }
    gate_all = np.ones(n_cases, dtype=bool)
    for values in gate_arrays.values():
        gate_all &= values
    result = {
        "fusion_W": fusion_W,
        "fusion_GW_module": fusion_W / 1.0e9,
        "alpha_W": alpha_W,
        "neutron_W": neutron_W,
        "alpha_deposited_W": alpha_deposited_W,
        "brem_W": brem_W,
        "transport_W": transport_W,
        "laser_optical_W": laser_optical_W,
        "laser_absorbed_W": laser_absorbed_W,
        "target_gain": target_gain,
        "self_heating_ratio": self_ratio,
        "beta_percent": 100.0 * beta,
        "captured_heat_W": captured_W,
        "captured_heat_GWth_module": captured_W / 1.0e9,
        "high_grade_W": high_grade_W,
        "low_grade_W": low_grade_W,
        "gross_GWe_hub": gross_hub_GWe,
        "laser_electric_GWe_hub": laser_electric_hub_GWe,
        "recirculating_GWe_hub": recirc_hub_GWe,
        "net_GWe_hub": net_hub_GWe,
        "T_burn_g_h_hub": t_burn_g_h,
        "T_surplus_g_h_hub": t_surplus_g_h,
        "tau_E_required_alpha_self_heating_s": tau_required,
        "tau_E_margin_ratio": tau_margin,
        "gate_all": gate_all,
    }
    for name, values in gate_arrays.items():
        result["gate_" + name] = values
    return result


def frame_to_params(frame):
    return {name: frame[name].to_numpy(dtype=np.float64) for name in PARAMETER_NAMES}


def evaluate_frame(frame, n_quad=N_QUAD):
    return pd.DataFrame(evaluate_batch(frame_to_params(frame), n_quad=n_quad))


def load_external_dataset(config):
    path = locate_file(config["filename"])
    frame = pd.read_csv(path)
    required = ["sample_id"] + PARAMETER_NAMES
    missing = [name for name in required if name not in frame.columns]
    if missing:
        raise ValueError(f"{config['filename']} is missing columns: {missing}")
    if frame.empty:
        raise ValueError(f"{config['filename']} is empty")
    if not frame["sample_id"].is_unique:
        raise ValueError(f"{config['filename']} has duplicate sample_id values")
    numeric = frame[PARAMETER_NAMES].to_numpy(dtype=np.float64)
    if not np.isfinite(numeric).all():
        raise ValueError(f"{config['filename']} contains non-finite inputs")
    dataset_ids = sorted(frame["dataset_id"].astype(str).unique().tolist()) if "dataset_id" in frame.columns else []
    if len(dataset_ids) > 1:
        raise ValueError(f"{config['filename']} contains multiple dataset_id values: {dataset_ids}")
    dataset_id = dataset_ids[0] if dataset_ids else config["role"]
    if "controlled_envelope" in config["role"] and (
        float(frame["TBR"].min()) < 1.15 or float(frame["wallplug_efficiency"].min()) < 0.16
    ):
        raise ValueError(f"{config['filename']} failed its controlled-envelope range signature")
    if "multifactor" in config["role"] and (
        float(frame["TBR"].min()) < 1.15 or float(frame["wallplug_efficiency"].min()) < 0.16
    ):
        raise ValueError(f"{config['filename']} failed its multifactor range signature")
    spec_path = find_optional_spec(path)
    return frame.reset_index(drop=True), {
        "path": str(path),
        "sha256": sha256_file(path),
        "dataset_id": dataset_id,
        "spec_path": str(spec_path) if spec_path else None,
        "spec_sha256": sha256_file(spec_path) if spec_path else None,
    }


def wilson_interval(successes, total, z=1.959963984540054):
    if total <= 0:
        return float("nan"), float("nan")
    p = successes / total
    denominator = 1.0 + z * z / total
    center = (p + z * z / (2.0 * total)) / denominator
    half = z * math.sqrt(p * (1.0 - p) / total + z * z / (4.0 * total * total)) / denominator
    return 100.0 * (center - half), 100.0 * (center + half)


def relative_error(actual, expected):
    actual = np.asarray(actual, dtype=np.float64)
    expected = np.asarray(expected, dtype=np.float64)
    return np.abs(actual - expected) / np.maximum(np.abs(expected), 1.0e-300)


def correlation(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if np.std(x) == 0.0 or np.std(y) == 0.0:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


check_rows = []


def record_check(name, passed, detail):
    check_rows.append({"check": name, "status": "PASS" if passed else "FAIL", "detail": detail})
    print(f"[{check_rows[-1]['status']}] {name}: {detail}")


# 1) Reference-rate verification against the Bosch-Hale table reproduced in
# UWFDM-1268, Table 6. The published table is in cm^3/s; this model uses m^3/s.
reference_T_keV = np.array([1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0])
reference_sigma_v_cm3_s = np.array([6.86e-21, 2.98e-19, 1.37e-17, 1.14e-16, 4.33e-16, 8.65e-16, 8.45e-16])
reference_sigma_v_m3_s = reference_sigma_v_cm3_s * 1.0e-6
reference_actual = bosch_hale_dt_reactivity(reference_T_keV)
reference_rel_error = relative_error(reference_actual, reference_sigma_v_m3_s)
record_check(
    "Bosch-Hale D-T reference table",
    bool(np.max(reference_rel_error) < 0.01),
    f"max relative error={np.max(reference_rel_error) * 100.0:.4f}% at published rounded values",
)

# 2) Exact accounting and implementation checks on a small deterministic probe.
probe_config = DATASETS[0]
probe_path = locate_file(probe_config["filename"])
probe_frame = pd.read_csv(probe_path).head(256).reset_index(drop=True)
probe_params = frame_to_params(probe_frame)
probe_result = evaluate_batch(probe_params, n_quad=N_QUAD)
record_check(
    "Fusion energy partition",
    bool(np.allclose(probe_result["alpha_W"] + probe_result["neutron_W"], probe_result["fusion_W"], rtol=1e-13, atol=1e-18)),
    "alpha plus neutron power equals total fusion power",
)
captured_expected = (
    probe_result["neutron_W"] * probe_params["neutron_heat_capture"]
    + probe_result["alpha_W"] * probe_params["alpha_heat_capture"]
    + probe_result["laser_absorbed_W"]
)
record_check(
    "Captured-heat accounting",
    bool(np.allclose(probe_result["captured_heat_W"], captured_expected, rtol=1e-13, atol=1e-18)),
    "neutron, alpha, and absorbed laser terms reconcile",
)
record_check(
    "Thermal-to-electric accounting",
    bool(np.allclose(probe_result["high_grade_W"] + probe_result["low_grade_W"], probe_result["captured_heat_W"], rtol=1e-13, atol=1e-18)),
    "high-grade plus low-grade heat equals captured heat",
)
record_check(
    "Finite and nonnegative core outputs",
    bool(np.isfinite(probe_result["fusion_W"]).all() and (probe_result["fusion_W"] >= 0.0).all()),
    "fusion output is finite and nonnegative for the probe",
)
record_check(
    "Laser reference power",
    bool(abs(2.50e6 * 10.0 / 1.0e6 - 25.0) < 1.0e-12),
    "2.50 MJ/module-shot at 10 Hz maps to 25 MW/module optical input",
)

# 3) Numerical convergence: the production screen uses N_QUAD nodes, while
# the audit compares 16/32/64-node radial quadrature on the same external rows.
q16 = evaluate_batch(probe_params, n_quad=16)
q32 = evaluate_batch(probe_params, n_quad=32)
q64 = evaluate_batch(probe_params, n_quad=64)
convergence_rows = []
for metric in ["fusion_W", "target_gain", "self_heating_ratio", "net_GWe_hub"]:
    err_16_32 = float(np.max(relative_error(q16[metric], q32[metric])))
    err_32_64 = float(np.max(relative_error(q32[metric], q64[metric])))
    convergence_rows.append({"metric": metric, "max_rel_16_to_32": err_16_32, "max_rel_32_to_64": err_32_64})
convergence_frame = pd.DataFrame(convergence_rows)
max_convergence_error = float(convergence_frame["max_rel_32_to_64"].max())
record_check(
    "Radial quadrature convergence",
    bool(max_convergence_error < 1.0e-4),
    f"max 32-to-64 relative change={max_convergence_error:.3e}",
)


summary_rows = []
gate_rows = []
sensitivity_rows = []
audit_frames = []
top_case_frames = []
dataset_manifest = []

for config in DATASETS:
    frame, file_meta = load_external_dataset(config)
    started = time.perf_counter()
    metrics = evaluate_frame(frame, n_quad=N_QUAD)
    elapsed = time.perf_counter() - started
    repeat_metrics = evaluate_frame(frame, n_quad=N_QUAD)
    repeat_columns = [column for column in metrics.columns if column != "gate_all"]
    repeat_exact = all(np.array_equal(metrics[column].to_numpy(), repeat_metrics[column].to_numpy()) for column in repeat_columns)
    repeat_exact = repeat_exact and np.array_equal(metrics["gate_all"].to_numpy(), repeat_metrics["gate_all"].to_numpy())
    record_check(
        f"Repeatability: {config['role']}",
        bool(repeat_exact),
        f"two evaluations of the same external CSV are bytewise-equivalent in output arrays; {len(frame):,} rows",
    )

    case_view = pd.concat([frame[["sample_id"]].reset_index(drop=True), metrics.reset_index(drop=True)], axis=1)
    if SAVE_FULL_CASE_OUTPUT:
        case_view.to_csv(OUT_DIR / f"{config['role']}_case_results.csv.gz", index=False, compression="gzip")
    audit_frames.append(pd.concat([frame.head(AUDIT_SAMPLE_PER_DATASET), metrics.head(AUDIT_SAMPLE_PER_DATASET)], axis=1).assign(dataset_role=config["role"]))
    top_case_frames.append(
        case_view[case_view["gate_all"]].sort_values("net_GWe_hub", ascending=False).head(100).assign(dataset_role=config["role"])
    )

    pass_count = int(metrics["gate_all"].sum())
    ci_low, ci_high = wilson_interval(pass_count, len(frame))
    summary_rows.append({
        "dataset_role": config["role"],
        "dataset_id": file_meta["dataset_id"],
        "description": config["description"],
        "input_file": config["filename"],
        "rows": int(len(frame)),
        "simultaneous_passes": pass_count,
        "pass_rate_percent": 100.0 * pass_count / len(frame),
        "wilson_95_low_percent": ci_low,
        "wilson_95_high_percent": ci_high,
        "elapsed_seconds": elapsed,
        "n_quadrature": N_QUAD,
        "input_sha256": file_meta["sha256"],
        "spec_sha256": file_meta["spec_sha256"],
    })
    for column in [column for column in metrics.columns if column.startswith("gate_") and column != "gate_all"]:
        gate_rows.append({
            "dataset_role": config["role"],
            "gate": column[5:],
            "passes": int(metrics[column].sum()),
            "pass_rate_percent": 100.0 * float(metrics[column].mean()),
        })
    output_targets = ["gate_all", "fusion_GW_module", "target_gain", "net_GWe_hub"]
    for parameter in PARAMETER_NAMES:
        for target in output_targets:
            sensitivity_rows.append({
                "dataset_role": config["role"],
                "parameter": parameter,
                "target": target,
                "pearson_correlation": correlation(frame[parameter].to_numpy(), metrics[target].astype(float).to_numpy()),
            })
    dataset_manifest.append({
        **config,
        **file_meta,
        "rows": int(len(frame)),
        "evaluated_seconds": elapsed,
        "repeated_output_identical": bool(repeat_exact),
    })


summary_frame = pd.DataFrame(summary_rows)
gate_frame = pd.DataFrame(gate_rows)
sensitivity_frame = pd.DataFrame(sensitivity_rows)
audit_frame = pd.concat(audit_frames, ignore_index=True)
top_cases_frame = pd.concat(top_case_frames, ignore_index=True) if top_case_frames else pd.DataFrame()
checks_frame = pd.DataFrame(check_rows)
convergence_frame.to_csv(OUT_DIR / "quadrature_convergence.csv", index=False)
summary_frame.to_csv(OUT_DIR / "validation_summary.csv", index=False)
gate_frame.to_csv(OUT_DIR / "validation_gate_rates.csv", index=False)
sensitivity_frame.to_csv(OUT_DIR / "validation_sensitivity.csv", index=False)
audit_frame.to_csv(OUT_DIR / "validation_audit_sample.csv", index=False)
top_cases_frame.to_csv(OUT_DIR / "validation_top_passing_cases.csv", index=False)
checks_frame.to_csv(OUT_DIR / "validation_checks.csv", index=False)

# Plot 1: scenario pass rates with 95% Wilson intervals.
plt.figure(figsize=(10, 5))
x = np.arange(len(summary_frame))
bars = plt.bar(x, summary_frame["pass_rate_percent"], color=["#6b7280", "#9ca3af", "#2563eb", "#16a34a"])
plt.errorbar(
    x,
    summary_frame["pass_rate_percent"],
    yerr=[summary_frame["pass_rate_percent"] - summary_frame["wilson_95_low_percent"], summary_frame["wilson_95_high_percent"] - summary_frame["pass_rate_percent"]],
    fmt="none",
    ecolor="black",
    capsize=4,
)
plt.xticks(x, [value.replace("_", "\n") for value in summary_frame["dataset_role"]], rotation=0)
plt.ylabel("All-gate pass rate (%)")
plt.title("Tomahawk Rev2.2 — external-input scenario results")
plt.ylim(0, 105)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "scenario_pass_rates.png", dpi=160)
plt.show()

# Plot 2: gate rates by scenario.
pivot = gate_frame.pivot(index="gate", columns="dataset_role", values="pass_rate_percent")
plt.figure(figsize=(12, 6))
pivot.plot(kind="bar", figsize=(12, 6), ax=plt.gca())
plt.ylabel("Gate pass rate (%)")
plt.xlabel("Gate")
plt.title("Individual gate rates; simultaneous pass is reported separately")
plt.ylim(0, 105)
plt.grid(axis="y", alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "individual_gate_rates.png", dpi=160)
plt.show()


def markdown_table(frame, columns, digits=3):
    rows = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for _, row in frame.iterrows():
        values = []
        for column in columns:
            value = row[column]
            if isinstance(value, (float, np.floating)):
                values.append(f"{float(value):.{digits}f}")
            else:
                values.append(str(value))
        rows.append("| " + " | ".join(values) + " |")
    return "\n".join(rows)


all_checks_pass = bool((checks_frame["status"] == "PASS").all())
report_lines = [
    "# Tomahawk Rev2.2 — Kaggle computational validation report",
    "",
    f"Run time (UTC): {datetime.now(timezone.utc).isoformat()}",
    f"Model revision: `{MODEL_REVISION}`",
    f"Model specification SHA-256: `{MODEL_SPEC_SHA256}`",
    "",
    "## Executive result",
    "",
    "This run verifies the frozen implementation, its numerical bookkeeping, its reference-rate implementation, and its repeatability on external CSV inputs. It does **not** establish that the reactor, target, confinement system, RF system, blanket, or power plant will work in hardware.",
    "",
    "The all-gate percentages below are conditional on the declared input scenarios. The broad and computational-holdout rows are stress/software checks. The controlled and multifactor rows are proposed design envelopes, not measured probability distributions.",
    "",
    markdown_table(summary_frame, ["dataset_role", "rows", "simultaneous_passes", "pass_rate_percent", "wilson_95_low_percent", "wilson_95_high_percent"]),
    "",
    "## Verification status",
    "",
    f"Implementation checks: **{'PASS' if all_checks_pass else 'FAIL'}**. A failure means the run should not be presented to investors until corrected.",
    "",
    markdown_table(checks_frame, ["check", "status", "detail"], digits=4),
    "",
    "## What this evidence supports",
    "",
    "- The same frozen evaluator gives identical output arrays when rerun on the same external CSV.",
    "- The Bosch-Hale D-T reactivity implementation reproduces the rounded reference table used for the audit within the stated tolerance.",
    "- Energy partition, captured-heat accounting, thermal-to-electric accounting, and radial quadrature are internally consistent.",
    "- The scenario results are traceable to input-file hashes, a declared model revision, row counts, and a fixed quadrature setting.",
    "",
    "## What it does not support",
    "",
    "- It is not experimental validation or independent hardware qualification.",
    "- Synthetic input ranges are not empirical uncertainty distributions, and their pass rate cannot be quoted as a probability that a reactor will operate.",
    "- The model does not solve 3-D MHD equilibrium/stability, target hydrodynamics, RF wave coupling, neutronics/TBR transport, structural FEA, materials lifetime, tritium systems, or plant commissioning risk.",
    "- The 60% threshold, if used, must be described as a pass rate inside a specified computational envelope, not as global feasibility or a field probability.",
    "",
    "## Recommended investor wording",
    "",
    "\"An independently rerunnable Kaggle audit verified the frozen Rev2.2 screening implementation against a published Bosch-Hale D-T reactivity table, conservation/accounting identities, quadrature convergence, repeatability, and external input datasets. The results are preliminary computational evidence and define the next engineering validation gates; they are not a claim of demonstrated reactor performance.\"",
    "",
    "## Source for the reaction-rate benchmark",
    "",
    "Bosch-Hale reference-rate values and the reproduced Table 6 are documented in [UWFDM-1268](https://fti.neep.wisc.edu/fti.neep.wisc.edu/pdf/fdm1268.pdf), which cites Bosch and Hale, *Nuclear Fusion* 32 (1992) 611. The original bibliographic record is available from [OSTI](https://www.osti.gov/etdeweb/biblio/5161054).",
]
for item in dataset_manifest:
    report_lines.extend([
        "",
        f"### Input: {item['filename']}",
        f"- Role: {item['role']}",
        f"- Dataset ID: {item['dataset_id']}",
        f"- Rows: {item['rows']}",
        f"- CSV SHA-256: `{item['sha256']}`",
        f"- Spec SHA-256: `{item['spec_sha256']}`",
    ])
report_path = OUT_DIR / "investor_validation_report.md"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

manifest = {
    "model_version": MODEL_VERSION,
    "model_revision": MODEL_REVISION,
    "model_spec_sha256": MODEL_SPEC_SHA256,
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "quadrature_nodes": N_QUAD,
    "save_full_case_output": SAVE_FULL_CASE_OUTPUT,
    "all_implementation_checks_pass": all_checks_pass,
    "datasets": dataset_manifest,
    "outputs": sorted(path.name for path in OUT_DIR.iterdir() if path.is_file()),
    "scope": "computational verification and conditional external-input scenario screening; not physical validation",
}
(OUT_DIR / "validation_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

print("\n=== Validation summary ===")
print(summary_frame[["dataset_role", "rows", "simultaneous_passes", "pass_rate_percent", "wilson_95_low_percent", "wilson_95_high_percent"]].to_string(index=False))
print("\nAll implementation checks pass:", all_checks_pass)
print("Outputs:", OUT_DIR)
